# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² Mental Health Kilifi County Dataset](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

This dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and standardized assessments (GAD-7, PHQ-9, PSQ) for psychological symptoms.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and overview from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset metadata (name, description, version)
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review the available record sets, their `@id`s, and included fields. All references to dataset components (record sets, fields, columns) are always via their `@id` field as per the Croissant schema.

In [ ]:
# Explore available record sets
record_sets = dataset.record_sets()
print(f"Record sets found (by @id):\n")
record_set_ids = []
for rs in record_sets:
    print(f"- {rs.id}: {rs.name if hasattr(rs, 'name') else 'No name'}")
    record_set_ids.append(rs.id)
print("\n")
# Show fields for each record set
for rs in dataset.record_sets():
    fields = rs.fields
    print(f"Fields in RecordSet {rs.id}:")
    for field in fields:
        desc = getattr(field, 'description', '')
        print(f"  - {field.id}: {field.name} (type={field.data_type}) {desc}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id`s identified previously.

In [ ]:
# Choose record sets and extract them as DataFrames (by @id)

# Usually, there's a main survey table. We'll print all available and pick the first one for this example.
main_record_set = record_set_ids[0]  # Use the first record set found
print(f"Using main record set: {main_record_set}")

dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f" - Loaded {len(dataframes[rs_id])} rows. Columns: {dataframes[rs_id].columns.tolist()}")
    else:
        print(f" - No records found.")

# Take a look at the first few rows from the main record set
if main_record_set in dataframes:
    print(f"\nFirst rows from main record set '{main_record_set}':")
    display(dataframes[main_record_set].head())
else:
    print(f"No DataFrame extracted for {main_record_set}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing a numeric field, or grouping. All fields are referenced by their `@id` field.

In [ ]:
# For illustration, let's pick a numeric field from the main table (e.g., total_psq_score or phq9_total)
# We'll auto-search for a likely numeric field
df = dataframes[main_record_set]
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields detected: {numeric_candidates}")

if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Use the first numeric field
    print(f"Using numeric field: {numeric_field}")
else:
    raise ValueError("No numeric fields found in data.")

# Filtering: Keep only records where this score is over 10
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalization
field_norm = f"{numeric_field}_normalized"
filtered_df[field_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} (z-score) for filtered records:")
display(filtered_df[[numeric_field, field_norm]].head())

# Grouping by a categorical field (try 'gender' or similar)
categorical_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None
# Heuristics to pick a reasonable categorical field
for col in ['gender', 'village', 'level_of_education', 'marital_status', 'education_level']:
    if col in categorical_fields:
        group_field = col
        break
if group_field:
    print(f"\nGrouping filtered results by '{group_field}':")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    display(grouped_df)
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the main numeric field filtered above, and show comparisons between groups (if available).

Here, we use `matplotlib` and `seaborn` for better aesthetics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field], bins=24, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field} (filtered > {threshold})")
    plt.show()

## 6. Conclusion
- We explored the FAIR² Kilifi Mental Health Survey dataset using the Croissant specification and `mlcroissant` library.
- Key variables (such as GAD-7, PHQ-9, PSQ scores) and demographic attributes are accessible by their Croissant `@id`s and were examined for their distributions and relationships.
- The schema supports accessing record sets and fields programmatically for reliable downstream AI data analysis tasks, with field provenance supported by the dataset metadata itself.
- For further research, refer to the [original dataset and Croissant schema documentation](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).